# handlers

> Workflow-specific alignment handlers: init

In [ ]:
#| default_exp alignment.routes.handlers

In [ ]:
#| export
from typing import Any, Dict, List, Tuple, Callable
from pathlib import Path

from fasthtml.common import Div, Span, APIRouter

from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_fasthtml_card_stack.core.models import CardStackState
from cjm_fasthtml_card_stack.core.constants import DEFAULT_VISIBLE_COUNT, DEFAULT_CARD_WIDTH

from cjm_fasthtml_workflow_transcript_decomp.core.html_ids import StructureDecompHtmlIds
from cjm_fasthtml_workflow_transcript_decomp.alignment.models import VADChunk, AlignmentUrls

# Alignment components (same package)
from cjm_fasthtml_workflow_transcript_decomp.alignment.components.card_stack_config import (
    ALIGN_CS_CONFIG, ALIGN_CS_IDS,
)
from cjm_fasthtml_workflow_transcript_decomp.alignment.components.step_renderer import (
    render_align_toolbar, render_align_stats, render_align_column_body,
    render_align_footer_content, render_align_mini_stats_text,
)
from cjm_fasthtml_workflow_transcript_decomp.components.step_combined import (
    render_alignment_status,
)

# Route core helpers (same package)
from cjm_fasthtml_workflow_transcript_decomp.alignment.routes.core import (
    _load_alignment_context, _update_alignment_state,
    _to_vad_chunks, _build_card_stack_state,
)
from cjm_fasthtml_workflow_transcript_decomp.alignment.routes.card_stack import (
    _build_nav_response,
)

from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import StructureDecompWorkflow

# Debug flag for alignment data flow tracing (set False in production)
DEBUG_ALIGNMENT = True

## Initialize Handler

Fetches VAD data from the alignment service and stores in state.

In [ ]:
#| export
async def _handle_align_init(
    workflow:Any,  # StructureDecompWorkflow instance
    request:Any,  # FastHTML request object
    sess:Any,  # FastHTML session object
    urls:AlignmentUrls,  # URL bundle
    visible_count:int=5,  # Initial visible card count
    card_width:int=40,  # Initial card width in rem
) -> tuple:  # (column_body, mini_stats_oob, alignment_status_oob)
    """Initialize alignment from audio file via VAD plugin.
    
    Note: KB system is always built by decomp init handler. This handler
    only returns column body, mini-stats OOB, and alignment status OOB.
    """
    session_id = get_session_id(sess)

    if DEBUG_ALIGNMENT:
        print(f"[ALIGN_INIT] Starting alignment initialization")
        print(f"[ALIGN_INIT] session_id: {session_id}")

    # Get media_path from selected sources (Phase 1)
    state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)
    step_states = state.get("step_states", {})
    selection_state = step_states.get("selection", {})
    selected_sources = selection_state.get("selected_sources", [])
    
    # Get segment count from decomposition state for alignment status
    decomp_state = step_states.get("decomposition", {})
    segment_count = len(decomp_state.get("segments", []))

    if DEBUG_ALIGNMENT:
        print(f"[ALIGN_INIT] selected_sources count: {len(selected_sources)}")
        print(f"[ALIGN_INIT] segment_count: {segment_count}")

    # Extract media_path from first selected source's source block
    media_path = None
    if selected_sources:
        first_source = selected_sources[0]
        block = workflow.source_service.get_transcription_by_id(
            record_id=first_source["record_id"],
            provider_id=first_source["provider_id"],
        )
        if DEBUG_ALIGNMENT and block:
            print(f"[ALIGN_INIT] block.media_path: {block.media_path}")
        if block:
            media_path = block.media_path

    if DEBUG_ALIGNMENT:
        print(f"[ALIGN_INIT] extracted media_path: {media_path}")

    # Fetch VAD data
    chunks = []
    audio_duration = 0.0
    if media_path and workflow.alignment_service.is_available():
        if DEBUG_ALIGNMENT:
            print(f"[ALIGN_INIT] Calling VAD plugin with media_path: {media_path}")
        chunks, audio_duration = await workflow.alignment_service.analyze_audio_async(media_path)
        if DEBUG_ALIGNMENT:
            print(f"[ALIGN_INIT] VAD returned {len(chunks)} chunks, duration: {audio_duration:.2f}s")

    chunk_count = len(chunks)

    # Serialize and store
    chunk_dicts = [c.to_dict() for c in chunks]
    _update_alignment_state(
        workflow, session_id,
        vad_chunks=chunk_dicts,
        focused_chunk_index=0,
        is_initialized=True,
        visible_count=visible_count,
        card_width=card_width,
        media_path=media_path,
        audio_duration=audio_duration,
    )

    # Render column body (Web Audio API handles accurate seeking)
    column_body = render_align_column_body(
        chunks=chunks,
        focused_index=0,
        visible_count=visible_count,
        card_width=card_width,
        urls=urls,
        kb_system=None,
        media_path=media_path,
    )

    # Mini-stats badge OOB update for the column header
    mini_stats_oob = Span(
        render_align_mini_stats_text(chunks),
        id=StructureDecompHtmlIds.ALIGNMENT_MINI_STATS,
        hx_swap_oob="true"
    )
    
    # Alignment status OOB update
    alignment_status_oob = render_alignment_status(
        segment_count=segment_count,
        chunk_count=chunk_count,
        oob=True,
    )

    if DEBUG_ALIGNMENT:
        print(f"[ALIGN_INIT] Returning column_body + mini_stats_oob + alignment_status_oob")

    return (column_body, mini_stats_oob, alignment_status_oob)

## Router Initialization

Creates the workflow router with the init route.

In [ ]:
#| export
def init_workflow_router(
    workflow: StructureDecompWorkflow,  # The workflow instance
    prefix: str,  # Route prefix (e.g., "/workflow/align/workflow")
    urls: AlignmentUrls,  # URL bundle (populated after routes defined)
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    """Initialize workflow routes for alignment."""
    router = APIRouter(prefix=prefix)

    # -------------------------------------------------------------------------
    # Workflow Operations
    # -------------------------------------------------------------------------

    @router
    async def init(request, sess):
        """Initialize alignment from audio file via VAD plugin."""
        return await _handle_align_init(workflow, request, sess, urls=urls)

    # -------------------------------------------------------------------------
    # Route Dict
    # -------------------------------------------------------------------------

    routes = {
        "init": init,
    }

    return router, routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()